# Diet, Income & BMI in US Adults (NHANES 2017–March 2020)

**Question:** How does what US adults eat differ by gender and income, and does daily calorie intake actually track with BMI?

**Data:** CDC's National Health and Nutrition Examination Survey (NHANES), pre-pandemic cycle 2017–March 2020. Four public, de-identified files are joined on the participant ID `SEQN`:

| File | Contents | Rows |
|---|---|---|
| `P_DEMO` | Demographics: age, gender, ethnicity, education, income-to-poverty ratio | 15,560 |
| `P_DR1TOT` | Day 1 dietary recall: total nutrient intake (24-hour interview) | 14,300 |
| `P_DR2TOT` | Day 2 dietary recall: phone follow-up 3–10 days later | 14,300 |
| `P_BMX` | Body measures: measured height, weight, BMI, waist | 14,300 |

**Tools:** pandas for wrangling, Bokeh for interactive charts.

## 1. Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.layouts import column
from bokeh.models import (ColumnDataSource, HoverTool, Select, CustomJS, DataTable,
                          TableColumn, NumberFormatter, LinearColorMapper, ColorBar,
                          LabelSet, Legend, Div)
from bokeh.transform import transform
from bokeh.palettes import Blues256, RdBu11

output_notebook(hide_banner=True)
pd.set_option("display.float_format", "{:,.1f}".format)

RAW = Path("data/raw"); RAW.mkdir(parents=True, exist_ok=True)
CDC = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/{}.xpt"

def load(name):
    """Read a cached copy from data/raw, otherwise download the XPT from the CDC and cache it."""
    cached = RAW / f"{name}.csv"
    if cached.exists():
        return pd.read_csv(cached, low_memory=False)
    df = pd.read_sas(CDC.format(name), format="xport")
    df.to_csv(cached, index=False)
    return df

## 2. Load the four files

In [2]:
demo = load("P_DEMO")
day1 = load("P_DR1TOT")
day2 = load("P_DR2TOT")
bmx  = load("P_BMX")

for name, df in [("Demographics", demo), ("Day 1 diet", day1), ("Day 2 diet", day2), ("Body measures", bmx)]:
    print(f"{name:<14} {df.shape[0]:>6,} rows  {df.shape[1]:>4} cols   unique SEQN: {df.SEQN.nunique():,}")

Demographics   15,560 rows    29 cols   unique SEQN: 15,560
Day 1 diet     14,300 rows   168 cols   unique SEQN: 14,300
Day 2 diet     14,300 rows    85 cols   unique SEQN: 14,300
Body measures  14,300 rows    22 cols   unique SEQN: 14,300


Every file has one row per participant, so `SEQN` is a clean one-to-one join key. Demographics has more rows because everyone interviewed is in it, but not everyone went on to the exam and diet interview.

## 3. Select and rename the variables used

NHANES column names are coded (`DR1TKCAL` = Day 1 total kcal). I keep only what the analysis needs and give it readable names.

In [3]:
demo_cols = {"SEQN": "seqn", "RIAGENDR": "gender", "RIDAGEYR": "age", "RIDRETH3": "ethnicity",
             "DMDEDUC2": "education", "INDFMPIR": "income_ratio"}
diet_cols = {"SEQN": "seqn", "WTDRD1PP": "diet_weight", "DR1DRSTZ": "recall_status",
             "DR1TKCAL": "kcal", "DR1TPROT": "protein_g", "DR1TCARB": "carbs_g", "DR1TSUGR": "sugar_g",
             "DR1TFIBE": "fibre_g", "DR1TTFAT": "fat_g", "DR1TSFAT": "sat_fat_g", "DR1TSODI": "sodium_mg"}
day2_cols = {"SEQN": "seqn", "DR2DRSTZ": "recall_status_d2", "DR2TKCAL": "kcal_d2"}
bmx_cols  = {"SEQN": "seqn", "BMXBMI": "bmi", "BMXWAIST": "waist_cm"}

df = (demo[list(demo_cols)].rename(columns=demo_cols)
      .merge(day1[list(diet_cols)].rename(columns=diet_cols), on="seqn", how="left")
      .merge(day2[list(day2_cols)].rename(columns=day2_cols), on="seqn", how="left")
      .merge(bmx[list(bmx_cols)].rename(columns=bmx_cols), on="seqn", how="left"))

# Decode categorical codes (NHANES codebook)
df["gender"] = df["gender"].map({1: "Men", 2: "Women"})
df["ethnicity"] = df["ethnicity"].map({1: "Mexican American", 2: "Other Hispanic", 3: "Non-Hispanic White",
                                       4: "Non-Hispanic Black", 6: "Non-Hispanic Asian", 7: "Other / Multiracial"})
df["education"] = df["education"].map({1: "Less than 9th grade", 2: "9-11th grade", 3: "High school / GED",
                                       4: "Some college", 5: "College graduate"})  # 7/9 = refused/don't know -> NaN
df["education"] = pd.Categorical(df["education"], ordered=True, categories=[
    "Less than 9th grade", "9-11th grade", "High school / GED", "Some college", "College graduate"])
df["seqn"] = df["seqn"].astype(int)
print(df.shape)
df[df.age >= 20].head()

(15560, 20)


,seqn,gender,age,ethnicity,education,income_ratio,diet_weight,recall_status,kcal,protein_g,carbs_g,sugar_g,fibre_g,fat_g,sat_fat_g,sodium_mg,recall_status_d2,kcal_d2,bmi,waist_cm
3,109266,Women,29.0,Non-Hispanic Asian,College graduate,5.0,"6,831.1",1.0,"1,698.0",52.6,217.7,94.2,20.7,73.8,22.2,"2,001.0",1.0,"1,896.0",37.8,117.9
4,109267,Women,21.0,Other Hispanic,Some college,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,109271,Men,49.0,Non-Hispanic White,9-11th grade,NaN,"15,526.9",1.0,"2,310.0",110.6,207.5,85.0,9.5,113.8,39.2,"5,446.0",1.0,"3,989.0",29.7,120.4
10,109273,Men,36.0,Non-Hispanic White,Some college,0.8,"40,576.8",1.0,"1,403.0",56.7,265.6,162.8,6.2,13.7,2.7,"1,990.0",1.0,"2,129.0",21.9,86.8
11,109274,Men,68.0,Other / Multiracial,Some college,1.2,"12,294.2",1.0,"2,385.0",77.8,305.4,60.8,27.7,86.4,21.8,"6,717.0",1.0,"1,302.0",30.2,109.6


## 4. Missing data: filter, don't fill

Missing values here are not random noise, they mean a step of the survey didn't happen (no diet interview, no exam, a child too young for an education question). Filling them with 0 would record people who skipped the diet interview as eating 0 calories and drag every average down, so instead I define the analysis population explicitly:

In [4]:
steps = [("All participants", df)]
a = df[df["age"] >= 20];                       steps.append(("Adults 20+", a))
a = a[a["recall_status"] == 1];                steps.append(("Complete, reliable Day 1 diet recall", a))
a = a[a["bmi"].notna()];                       steps.append(("Measured BMI", a))
a = a[a["diet_weight"] > 0].copy();            steps.append(("Valid dietary survey weight", a))

pd.DataFrame([(s, len(d)) for s, d in steps], columns=["Filter", "Rows remaining"])

,Filter,Rows remaining
0,All participants,15560
1,Adults 20+,9232
2,"Complete, reliable Day 1 diet recall",7707
3,Measured BMI,7613
4,Valid dietary survey weight,7613


In [5]:
# What's still missing in the analysis set?
a.isna().sum()[lambda s: s > 0].to_frame("missing")

,missing
education,8
income_ratio,957
kcal_d2,1042
waist_cm,175


Only income ratio (~10%, people who didn't report income) and education (a handful of refusals) are still missing. Those rows stay in, and they're excluded only from the charts that group by income or education.

**Day 2 as a sanity check.** The Day 2 recall shows how much one person's intake swings from day to day:

In [6]:
both = a[a["recall_status_d2"] == 1]
r = both[["kcal", "kcal_d2"]].corr().iloc[0, 1]
print(f"{len(both):,} adults completed both days. Day 1 vs Day 2 calorie correlation: r = {r:.2f}")

6,571 adults completed both days. Day 1 vs Day 2 calorie correlation: r = 0.44


A correlation of only ~0.4 between two days for the *same person* means a single 24-hour recall is a noisy measure of someone's usual diet. Group averages are reliable, but individual-level relationships (like calories vs BMI below) will look weaker than they really are.

**Survey weights.** NHANES deliberately oversamples some groups, so a plain average doesn't represent the US population. All averages below use the Day 1 dietary weight `WTDRD1PP`.

In [7]:
def wmean(g, col):
    m = g[col].notna()
    return np.average(g.loc[m, col], weights=g.loc[m, "diet_weight"])

def wmeans(g, cols):
    return pd.Series({c: wmean(g, c) for c in cols} | {"n": len(g)})

a["bmi_cat"] = pd.cut(a["bmi"], [0, 18.5, 25, 30, np.inf], right=False,
                      labels=["Underweight", "Healthy", "Overweight", "Obese"])
a["income_group"] = pd.cut(a["income_ratio"], [-np.inf, 1.3, 3.5, np.inf], right=False,
                           labels=["Low (<1.3x poverty line)", "Middle (1.3-3.5x)", "High (3.5x+)"])
a["age_group"] = pd.cut(a["age"], [20, 40, 60, 81], right=False, labels=["20-39", "40-59", "60+"])

share = a.groupby("bmi_cat", observed=True)["diet_weight"].sum() / a["diet_weight"].sum() * 100
share.round(1).to_frame("% of US adults (weighted)")

,% of US adults (weighted)
bmi_cat,
Underweight,1.4
Healthy,24.5
Overweight,32.0
Obese,42.1


**Validation:** the weighted obesity rate comes out at ~42%, which matches the CDC's published figure for this survey cycle (41.9%). That confirms the merge, filters and weights are working as intended.

---
## 5. Analysis & visualisation
### Chart 1: How men's and women's diets differ

In [8]:
nutrients = {"carbs_g": "Carbohydrates", "sugar_g": "Total sugars", "fat_g": "Total fat",
             "protein_g": "Protein", "sat_fat_g": "Saturated fat", "fibre_g": "Fibre"}
by_gender = a.groupby("gender").apply(lambda g: wmeans(g, list(nutrients) + ["kcal"]), include_groups=False)
by_gender[["kcal"] + list(nutrients)].T

gender,Men,Women
kcal,"2,483.8","1,832.0"
carbs_g,279.7,211.5
sugar_g,119.5,92.6
fat_g,101.0,76.4
protein_g,94.3,68.9
sat_fat_g,32.8,24.8
fibre_g,18.1,15.3


In [9]:
g = by_gender[list(nutrients)].T.rename(index=nutrients)
g["gap"] = g["Men"] - g["Women"]
g["pct_more"] = (g["Men"] / g["Women"] - 1) * 100
g = g.sort_values("Men")
src = ColumnDataSource(g.reset_index(names="nutrient"))

MEN, WOMEN = "#2b6cb0", "#dd6b20"
p1 = figure(y_range=list(g.index), height=380, width=760, x_axis_label="Average daily intake (grams)",
            title="Men eat more of everything, but fibre is low for both",
            tools="pan,wheel_zoom,reset", toolbar_location="right")
p1.segment(x0="Women", x1="Men", y0="nutrient", y1="nutrient", source=src, line_width=3, color="#cbd5e0")
r_w = p1.scatter("Women", "nutrient", source=src, size=13, color=WOMEN)
r_m = p1.scatter("Men", "nutrient", source=src, size=13, color=MEN)
p1.add_layout(Legend(items=[("Men", [r_m]), ("Women", [r_w])], location="bottom_right"))
p1.add_tools(HoverTool(renderers=[r_m, r_w], tooltips=[("Nutrient", "@nutrient"), ("Men", "@Men{0.0} g"),
            ("Women", "@Women{0.0} g"), ("Men eat", "@pct_more{0}% more")]))
p1.x_range.start = 0
p1.ygrid.grid_line_color = None
p1.toolbar.autohide = True
show(p1)

- Men report ~2,480 kcal/day vs ~1,830 for women (+36%), and they eat 29–37% more of every macronutrient, which mostly reflects higher energy needs.
- **Fibre is the standout:** men average ~18 g and women ~15 g, both well under the US guideline of ~28 g per 2,000 kcal. Fibre has the smallest gap (men eat only 18% more), so both genders fall short by a similar margin.

### Chart 2: Does calorie intake track with BMI?

In [10]:
h = a[(a["kcal"].between(500, 6000)) & (a["bmi"] < 70)]
rho = h[["kcal", "bmi"]].corr(method="spearman").iloc[0, 1]
print(f"Spearman correlation, BMI vs reported calories: {rho:.2f}  (n = {len(h):,})")
h.groupby("bmi_cat", observed=True).apply(lambda g: wmean(g, "kcal"), include_groups=False).round(0).to_frame("avg kcal")

Spearman correlation, BMI vs reported calories: -0.00  (n = 7,481)


,avg kcal
bmi_cat,
Underweight,"2,264.0"
Healthy,"2,118.0"
Overweight,"2,130.0"
Obese,"2,159.0"


In [11]:
p2 = figure(width=760, height=480, background_fill_color="#f7fafc", x_range=(13, 65), y_range=(3, 62),
            title=f"No relationship between reported calories and BMI (Spearman ρ = {abs(rho):.2f})",
            x_axis_label="BMI (measured)", y_axis_label="Reported calories (Day 1, kcal)",
            tools="pan,wheel_zoom,reset")
p2.grid.visible = False
r, bins = p2.hexbin(h["bmi"], h["kcal"] / 100, size=1.0, line_color="white", line_alpha=0.6,
                    palette=Blues256[::-1][40:])
# hexbin needs comparable axis scales, so calories are binned in hundreds and the axis relabelled
p2.yaxis.major_label_overrides = {i: f"{i*100:,}" for i in range(0, 61, 10)}
p2.add_tools(HoverTool(renderers=[r], tooltips=[("Adults in this cell", "@c")]))
for x in (18.5, 25, 30):
    p2.ray(x=x, y=3, length=0, angle=np.pi/2, line_dash="dashed", line_color="#718096")
for x, lab in [(21.75, "Healthy"), (27.5, "Over-"), (34, "Obese →")]:
    p2.text(x=[x], y=[59], text=[lab], text_font_size="9pt", text_color="#4a5568", text_align="center")
p2.text(x=[27.5], y=[56.5], text=["weight"], text_font_size="9pt", text_color="#4a5568", text_align="center")
p2.toolbar.autohide = True
show(p2)

- The densest region is BMI 23–33 at 1,500–2,500 kcal, and the cloud is **flat**: average reported intake is ~2,100–2,200 kcal in every BMI category.
- This doesn't mean calories don't matter for weight. Known measurement issues in diet surveys explain it:
  1. **Under-reporting** of food intake rises with BMI (well documented in NHANES research).
  2. **One day ≠ usual diet:** the r ≈ 0.44 day-to-day correlation above shows how noisy a single recall is.
  3. **Reverse causation:** people with higher BMI are more likely to be dieting on the recall day.
- **Takeaway:** self-reported one-day calories are a poor individual-level predictor of BMI, so measured outcomes need different evidence.

### Chart 3: Diet quality by household income

In [12]:
inc_cols = {"sugar_g": "Total sugars (g)", "fibre_g": "Fibre (g)", "kcal": "Calories (kcal)",
            "sat_fat_g": "Saturated fat (g)", "sodium_mg": "Sodium (mg)", "bmi": "BMI"}
by_income = (a.dropna(subset=["income_group"]).groupby("income_group", observed=True)
               .apply(lambda g: wmeans(g, list(inc_cols)), include_groups=False))
by_income

,sugar_g,fibre_g,kcal,sat_fat_g,sodium_mg,bmi,n
income_group,,,,,,,
Low (<1.3x poverty line),118.7,15.4,"2,151.3",27.4,"3,385.7",30.0,"1,857.0"
Middle (1.3-3.5x),109.9,16.2,"2,145.7",28.7,"3,462.3",30.4,"2,592.0"
High (3.5x+),96.5,17.3,"2,144.5",29.4,"3,513.9",29.4,"2,207.0"


In [13]:
groups = [str(x) for x in by_income.index]
data = {"group": groups, **{k: by_income[k].round(1).tolist() for k in inc_cols}, "value": by_income["sugar_g"].round(1).tolist()}
src3 = ColumnDataSource(data)

p3 = figure(x_range=groups, height=420, width=760, title="Average daily intake by household income",
            y_axis_label="Total sugars (g)", tools="", toolbar_location=None)
p3.vbar(x="group", top="value", source=src3, width=0.6, color="#2c7a7b")
p3.add_layout(LabelSet(x="group", y="value", text="value", source=src3, text_align="center",
                       y_offset=4, text_font_size="11pt"))
p3.add_tools(HoverTool(tooltips=[("Income", "@group"), ("Value", "@value{0,0.0}")]))
p3.y_range.start = 0
p3.xgrid.grid_line_color = None

sel = Select(title="Measure:", value="sugar_g", options=[(k, v) for k, v in inc_cols.items()], width=250)
sel.js_on_change("value", CustomJS(args=dict(src=src3, sel=sel, ax=p3.yaxis[0], labels=dict(inc_cols)), code="""
    const d = src.data;
    d.value = d[sel.value].slice();
    ax.axis_label = labels[sel.value];
    src.change.emit();
"""))
layout3 = column(sel, p3)
show(layout3)

In [14]:
low, high = by_income.iloc[0], by_income.iloc[-1]
print(f"Sugar: {low.sugar_g:.0f} g -> {high.sugar_g:.0f} g ({(high.sugar_g/low.sugar_g-1)*100:+.0f}%)")
print(f"Fibre: {low.fibre_g:.1f} g -> {high.fibre_g:.1f} g ({(high.fibre_g/low.fibre_g-1)*100:+.0f}%)")
print(f"Calories: {low.kcal:.0f} -> {high.kcal:.0f} kcal")

Sugar: 119 g -> 97 g (-19%)
Fibre: 15.4 g -> 17.3 g (+12%)
Calories: 2151 -> 2145 kcal


- Total calories are almost identical across income groups (~2,150 kcal), but **what** people eat differs. Higher-income adults eat ~19% less sugar and ~12% more fibre than low-income adults.
- So the income gap is about diet *quality*, not quantity. This fits the social-determinants-of-health view: cost, access and time shape food choices more than the amount eaten.
- Not everything improves with income: sodium and saturated fat are slightly *higher* in the high-income group (use the dropdown to switch measures).
- Average BMI is lowest in the high-income group (29.4 vs 30.0–30.4).

### Chart 4: Explore any demographic group (interactive table)

In [15]:
dims = {"gender": "Gender", "age_group": "Age group", "income_group": "Income", "education": "Education",
        "ethnicity": "Ethnicity"}
rows = []
for col, label in dims.items():
    for grp, g in a.dropna(subset=[col]).groupby(col, observed=True):
        obese = g.loc[g.bmi_cat == "Obese", "diet_weight"].sum() / g["diet_weight"].sum() * 100
        rows.append({"dimension": label, "group": str(grp), "n": len(g), "kcal": wmean(g, "kcal"),
                     "sugar": wmean(g, "sugar_g"), "fibre": wmean(g, "fibre_g"),
                     "sodium": wmean(g, "sodium_mg"), "bmi": wmean(g, "bmi"), "obese_pct": obese})
summary = pd.DataFrame(rows)
summary.to_csv("data/group_summary.csv", index=False)
summary.round(1)

,dimension,group,n,kcal,sugar,fibre,sodium,bmi,obese_pct
0,Gender,Men,3697,"2,483.8",119.5,18.1,"4,023.2",29.6,42.6
1,Gender,Women,3916,"1,832.0",92.6,15.3,"2,951.7",29.9,41.6
2,Age group,20-39,2345,"2,235.6",107.7,16.6,"3,709.2",29.3,40.7
3,Age group,40-59,2529,"2,192.1",110.7,16.8,"3,460.9",30.4,44.3
4,Age group,60+,2739,"1,981.4",97.0,16.5,"3,178.5",29.6,41.2
5,Income,Low (<1.3x poverty line),1857,"2,151.3",118.7,15.4,"3,385.7",30.0,43.9
6,Income,Middle (1.3-3.5x),2592,"2,145.7",109.9,16.2,"3,462.3",30.4,45.8
7,Income,High (3.5x+),2207,"2,144.5",96.5,17.3,"3,513.9",29.4,40.0
8,Education,Less than 9th grade,514,"2,020.1",99.8,19.9,"3,261.5",29.9,41.8
9,Education,9-11th grade,815,"2,183.8",119.1,15.3,"3,393.0",29.8,41.8


In [16]:
full = ColumnDataSource(summary)
view = ColumnDataSource(summary[summary.dimension == "Education"])
f1, f0 = NumberFormatter(format="0.0"), NumberFormatter(format="0,0")
cols = [TableColumn(field="group", title="Group", width=210),
        TableColumn(field="n", title="Sample size", formatter=f0),
        TableColumn(field="kcal", title="Calories", formatter=f0),
        TableColumn(field="sugar", title="Sugar (g)", formatter=f1),
        TableColumn(field="fibre", title="Fibre (g)", formatter=f1),
        TableColumn(field="sodium", title="Sodium (mg)", formatter=f0),
        TableColumn(field="bmi", title="Avg BMI", formatter=f1),
        TableColumn(field="obese_pct", title="% obese", formatter=f1)]
for c in cols[1:]:
    c.width = 90
table = DataTable(source=view, columns=cols, width=880, height=210, index_position=None, sortable=True,
                  autosize_mode="none")
pick = Select(title="Break down by:", value="Education", options=list(dims.values()), width=250)
pick.js_on_change("value", CustomJS(args=dict(full=full, view=view, pick=pick), code="""
    const f = full.data, out = {};
    for (const k of Object.keys(f)) out[k] = [];
    for (let i = 0; i < f.dimension.length; i++)
        if (f.dimension[i] === pick.value)
            for (const k of Object.keys(f)) out[k].push(f[k][i]);
    view.data = out;
"""))
note = Div(text="<i>Weighted averages, US adults 20+, NHANES 2017–Mar 2020. Click a column header to sort.</i>")
layout4 = column(pick, table, note)
show(layout4)

- **Education shows the sharpest split:** college graduates have the lowest sugar intake (96 g), among the highest fibre (19 g), and an obesity rate of 34%, compared with 45–47% for adults with a high-school diploma or some college.
- **Age:** adults 60+ report ~250 fewer calories than 20–39 year olds, yet their average BMI is no lower (29.6 vs 29.3). This is another sign that one-day calories don't map neatly onto body size.
- **Ethnicity:** obesity ranges from 16% (Non-Hispanic Asian) to over 50% in some groups. These differences overlap heavily with income and education, and standard BMI cut-offs are known to underestimate health risk in Asian populations, so I don't read them as causal.

### Chart 5: Which measures move together?

In [17]:
corr_vars = {"age": "Age", "income_ratio": "Income ratio", "bmi": "BMI", "waist_cm": "Waist",
             "kcal": "Calories", "protein_g": "Protein", "fat_g": "Fat", "sugar_g": "Sugar",
             "fibre_g": "Fibre", "sodium_mg": "Sodium"}
corr = a[list(corr_vars)].rename(columns=corr_vars).corr(method="spearman").round(2)
names = list(corr.columns)
cdf = corr.rename_axis(index="y", columns="x").stack().rename("r").reset_index()
cdf["label"] = cdf["r"].map("{:.2f}".format)
cdf["text_color"] = np.where(cdf["r"].abs() > 0.6, "white", "#1a202c")
src5 = ColumnDataSource(cdf)

mapper = LinearColorMapper(palette=RdBu11[::-1], low=-1, high=1)
p5 = figure(x_range=names, y_range=names[::-1], width=720, height=640, x_axis_location="above",
            title="Spearman correlations (adults 20+)", tools="", toolbar_location=None)
p5.rect("x", "y", 1, 1, source=src5, fill_color=transform("r", mapper), line_color="white")
p5.text("x", "y", text="label", source=src5, text_align="center", text_baseline="middle",
        text_font_size="9pt", text_color="text_color")
p5.add_layout(ColorBar(color_mapper=mapper, width=12, title="r"), "right")
p5.add_tools(HoverTool(tooltips=[("", "@y × @x"), ("r", "@r")]))
p5.xaxis.major_label_orientation = 0.8
p5.axis.axis_line_color = None; p5.axis.major_tick_line_color = None; p5.grid.visible = False
show(p5)

- **Two separate clusters:** body size (BMI ↔ waist, ρ = 0.91) and diet amount (calories ↔ fat 0.87, sodium 0.81, protein 0.78). The two clusters barely connect (ρ between −0.08 and 0.08), which matches Chart 2. Fibre is the only nutrient with a (weak) negative link to BMI.
- The nutrients correlate with each other mainly because people who eat more food eat more of everything. Comparing diet *quality* needs per-calorie measures.
- Income's strongest link (still small, ρ = 0.13) is with fibre, consistent with Chart 3.
- I used Spearman rather than Pearson because calories and sodium are heavily right-skewed.

---
## 6. Summary

| Finding | Evidence |
|---|---|
| ~42% of US adults are obese, ~32% overweight | Weighted BMI categories (matches CDC's published 41.9%) |
| Men eat ~35% more calories, but both genders get only about half the recommended fibre | Chart 1 |
| Reported calories don't predict BMI (ρ ≈ 0) | Chart 2, explained by under-reporting and day-to-day noise |
| Income changes diet **quality**, not **quantity**: −19% sugar, +12% fibre from low to high income, same calories | Chart 3 |
| College graduates: lowest sugar, high fibre, 34% obese vs 45–47% for high school / some college | Chart 4 |

## 7. Limitations & ethics
- **Cross-sectional data:** every relationship here is an association, not a cause.
- **Self-reported diet** is subject to recall bias and systematic under-reporting, especially at higher BMI.
- **Weighted means only.** A full survey-design analysis (using `SDMVPSU`/`SDMVSTRA` for standard errors) would be needed before making formal population claims.
- **Responsible comparison:** income, education and ethnicity differences reflect structural factors (food cost, access, time) rather than individual failings, and are presented that way to avoid stigmatising groups.
- **Privacy:** NHANES public files are de-identified by the CDC. No individual-level records are displayed.
- **US-only:** results may not generalise to Australia or elsewhere.